In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

In [2]:
train_df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/test.csv")

In [3]:
train_df.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


In [4]:
X = train_df.drop(columns=['id','Will_Buy_EV'])
Y = train_df['Will_Buy_EV']

In [5]:
X_train, X_valid, Y_train, Y_valid = train_test_split(X, Y, test_size=0.2,random_state=42)

In [6]:
print(Y_train.value_counts())
print(Y_train.value_counts(normalize=True))

Will_Buy_EV
No     441346
Yes     93586
Name: count, dtype: int64
Will_Buy_EV
No     0.825051
Yes    0.174949
Name: proportion, dtype: float64


In [7]:
scale_pos_weight = 441346 / 93586

In [8]:
cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(exclude='object').columns

In [9]:
print(cat_cols)
print(num_cols)

Index(['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
       'Subsidy_Available', 'Range_Anxiety_Level'],
      dtype='object')
Index(['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
       'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work',
       'Environmental_Concern_Level'],
      dtype='object')


In [10]:
cat_pipeline = Pipeline([
    ('LabelEncoder',OrdinalEncoder()),
])

preprocessor = ColumnTransformer([
    ('categoric',cat_pipeline,cat_cols)
],remainder='passthrough'
)

In [11]:
lgbm_1 = Pipeline([
    ('preprocessor', preprocessor),

    ('lgbm', LGBMClassifier(
        objective='binary',
        n_estimators=2500,
        learning_rate=0.02,

        num_leaves=31,
        max_depth=7,
        min_child_samples=40,

        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,

        reg_alpha=0.1,
        reg_lambda=1.5,

        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])

In [12]:
lgbm_2 = Pipeline([
    ('preprocessor', preprocessor),

    ('lgbm', LGBMClassifier(
        objective='binary',
        n_estimators=2500,
        learning_rate=0.02,

        num_leaves=15,
        max_depth=5,
        min_child_samples=60,

        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,

        reg_alpha=0.5,
        reg_lambda=3.0,

        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])

In [13]:
lgbm_3 = Pipeline([
    ('preprocessor', preprocessor),

    ('lgbm', LGBMClassifier(
        objective='binary',
        n_estimators=2500,
        learning_rate=0.015,

        num_leaves=63,
        max_depth=-1,
        min_child_samples=40,

        subsample=0.80,
        subsample_freq=1,
        colsample_bytree=0.80,

        reg_alpha=0.2,
        reg_lambda=2.0,

        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])

In [14]:
models = {
    "LightGBM 1": lgbm_1,
    "LightGBM 2": lgbm_2,
    "LightGBM 3": lgbm_3
}

results = {}
preds = {}

for name, model in models.items():

    print(f"\nTraining {name}...")

    model.fit(X_train, Y_train)

    valid_pred = model.predict_proba(X_valid)[:, 1]

    preds[name] = valid_pred

    auc = roc_auc_score(Y_valid, valid_pred)

    results[name] = auc

    print(f"{name} ROC-AUC: {auc:.6f}")


Training LightGBM 1...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM 1 ROC-AUC: 0.942105

Training LightGBM 2...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM 2 ROC-AUC: 0.942262

Training LightGBM 3...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM 3 ROC-AUC: 0.941918


In [15]:
results

{'LightGBM 1': np.float64(0.9421054526825899),
 'LightGBM 2': np.float64(0.9422616540832062),
 'LightGBM 3': np.float64(0.9419183910694839)}

In [16]:
# Run this in your second cell
ensemble_pred = sum(preds.values()) / len(preds)
ensemble_auc = roc_auc_score(Y_valid, ensemble_pred)

print(f"Ensemble ROC-AUC: {ensemble_auc:.6f}")


Ensemble ROC-AUC: 0.942163


In [17]:
lgbm_1.fit(X,Y)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('categoric',
                                                  Pipeline(steps=[('LabelEncoder',
                                                                   OrdinalEncoder())]),
                                                  Index(['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
       'Subsidy_Available', 'Range_Anxiety_Level'],
      dtype='object'))])),
                ('lgbm',
                 LGBMClassifier(colsample_bytree=0.85, learning_rate=0.02,
                                max_depth=7, min_child_samples=40,
                                n_estimators=2500, n_jobs=-1,
                                objective='binary', random_state=42,
                                reg_alpha=0.1, reg_lambda=1.5, subsample=0.85,
                                subsample_freq=1, verbosity=-1))])

In [18]:
y_preds = model.predict_proba(test_df.drop(columns=['id']))[:,1]

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [19]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'Will_Buy_EV':y_preds
})

submission.to_csv('submission.csv', index=False)